In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 1 — Load consolidated data from OneDrive
#
# First run: opens browser login (30 sec) → sign in with @consultdss.com
# All subsequent runs: fully automatic, no browser needed (90-day token)
# ═══════════════════════════════════════════════════════════════════

import sys
from pathlib import Path

# Make sure onedrive_loader.py is importable
# (it should be in the same folder as this notebook)
proj = Path(".").resolve()
if str(proj) not in sys.path:
    sys.path.insert(0, str(proj))

from onedrive_loader import load_consolidated, list_folder

# Show what files are in the OneDrive folder (verification step)
print("Files in OneDrive — Documents/DSS/Dummy data:")
try:
    for f in list_folder():
        icon = '📁' if f['type'] == 'folder' else '📄'
        size = f"{f['size']:,} bytes" if f.get('size') else ''
        print(f"  {icon}  {f['name']}  {size}")
except Exception as e:
    print(f"  (Could not list folder: {e})")

print()

# Load the consolidated file
# This will be used by all cells below instead of generated dummy data
df_onedrive = load_consolidated()


# TIP ESG Platform — Consolidation & Benchmarking Analysis
**dss+ | Tire Industry Project**

This notebook loads the full consolidated dataset, runs peer analysis, and generates all benchmarking charts.
Run after each new company submission is approved to refresh the analysis.

In [ ]:
import pandas as pd, numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings("ignore")

COMPANIES = ["VerdaTyres Corp","AlphaTread Ltd","BetaRubber Inc","GammaTire SA",
             "DeltaGrip GmbH","EpsilonWheel Co","ZetaTrac LLC","EtaRoad AG",
             "ThetaDrive NV","IotaTire PLC"]
YEARS = list(range(2009,2024))
print(f"Setup: {len(COMPANIES)} companies, {len(YEARS)} years, ~{len(COMPANIES)*len(YEARS)*46:,} data points")

## 1  Load / Generate Consolidated Dataset

In [ ]:
import pandas as pd, numpy as np

# Use OneDrive data if loaded successfully, otherwise generate dummy
if 'df_onedrive' in dir() and df_onedrive is not None and len(df_onedrive) > 0:
    df = df_onedrive.copy()
    
    # Normalise column names if coming from a different export format
    col_map = {
        "Production_MT":     "Prod_MT",
        "Total_Energy_GJ":   "Energy_GJ",
        "Total_CO2_T":       "CO2_T",
        "Waste_Total_MT":    "Waste_MT",
        "Waste_Recovery_MT": "Recovery_MT",
    }
    df = df.rename(columns={k: v for k, v in col_map.items() if k in df.columns})
    
    # Make sure required derived columns exist
    if 'Energy_KPI' not in df.columns and 'Energy_GJ' in df.columns:
        df['Energy_KPI'] = df['Energy_GJ'] / df['Prod_MT']
    if 'Water_KPI' not in df.columns and 'Water_M3' in df.columns:
        df['Water_KPI'] = df['Water_M3'] / df['Prod_MT']
    if 'Recovery_Pct' not in df.columns and 'Recovery_MT' in df.columns:
        df['Recovery_Pct'] = df['Recovery_MT'] / df['Waste_MT'] * 100
    
    print(f"Using OneDrive data: {len(df):,} rows")
else:
    # Generate built-in dummy data as fallback
    print("OneDrive data not available — generating dummy data")
    np.random.seed(42)

    def gen_company(name, scale):
        rng = np.random.RandomState(hash(name) % 2**31)
        prod  = [scale*1e6*(1+0.025*(y-2009)+rng.uniform(-0.04,0.04)) for y in YEARS]
        prod[11] *= 0.79
        energy = [p*rng.uniform(8.5,10.2) for p in prod]
        co2_s  = rng.uniform(0.68,0.92)
        co2_k  = [co2_s*(1-0.018*(y-2009)+rng.uniform(-0.015,0.015)) for y in YEARS]
        co2    = [p*k for p,k in zip(prod,co2_k)]
        water  = [p*rng.uniform(5.5,9.8) for p in prod]
        water[11] *= 0.83
        renew  = [max(0,(y-2013)*rng.uniform(2.5,6.5)) if y>2013 else 0 for y in YEARS]
        wt     = [p*rng.uniform(0.048,0.065) for p in prod]
        wr     = [w*rng.uniform(0.84,0.93) for w in wt]
        return pd.DataFrame({
            "Company":name,"Year":YEARS,
            "Prod_MT":prod,"Energy_GJ":energy,"CO2_T":co2,"CO2_KPI":co2_k,
            "Water_M3":water,"Renew_Pct":renew,
            "Waste_MT":wt,"Recovery_MT":wr,
            "Recovery_Pct":[r/t*100 for r,t in zip(wr,wt)],
            "Energy_KPI":[e/p for e,p in zip(energy,prod)],
            "Water_KPI":[w/p for w,p in zip(water,prod)],
        })

    SCALES = [1.51,2.6,1.8,1.65,0.9,1.2,0.72,0.62,0.76,0.42]
    df = pd.concat([gen_company(c,s) for c,s in zip(COMPANIES,SCALES)], ignore_index=True)

df['Year'] = df['Year'].astype(int)
print(f"Dataset: {len(df):,} rows x {len(df.columns)} columns")
display(df[df.Company=="VerdaTyres Corp"].tail(3)[[
    "Year","Prod_MT","Energy_GJ","CO2_T","CO2_KPI","Energy_KPI","Recovery_Pct"]].round(3))


## 2  Industry Overview Charts

In [ ]:
latest = df[df.Year==2023].copy()

fig = make_subplots(1,3,subplot_titles=["CO2 Intensity (T/T)","Energy Intensity (GJ/T)","Renewable Elec %"])
for col, row, ccol in [("CO2_KPI",True,"CO2_KPI"),("Energy_KPI",True,"Energy_KPI"),("Renew_Pct",False,"Renew_Pct")]:
    srt = latest.sort_values(col, ascending=row)
    colors = ["#00916E" if c=="VerdaTyres Corp" else "#9CA3AF" for c in srt.Company]
    c_idx = list(["CO2_KPI","Energy_KPI","Renew_Pct"]).index(ccol)+1
    fig.add_trace(go.Bar(x=srt.Company, y=srt[col].round(3),
        marker_color=colors, text=srt[col].round(2), textposition="outside"), 1, c_idx)

fig.update_xaxes(tickangle=-40, tickfont=dict(size=8))
fig.update_layout(height=430,showlegend=False,title_text="TIP Industry Comparison 2023",
    plot_bgcolor="white",paper_bgcolor="white")
fig.show()

## 3  Quartile Band Chart

In [ ]:
KPI_DEFS = [
    ("CO2 Intensity (T/T)",   "CO2_KPI",    True),
    ("Energy KPI (GJ/T)",     "Energy_KPI", True),
    ("Water KPI (m3/T)",      "Water_KPI",  True),
    ("Renewable Elec %",      "Renew_Pct",  False),
    ("Waste Recovery %",      "Recovery_Pct", False),
]
vt = latest[latest.Company=="VerdaTyres Corp"].iloc[0]

fig_b = go.Figure()
for label, col, lb in KPI_DEFS:
    vals = latest[col].values
    q25,med,q75 = np.percentile(vals,[25,50,75])
    val = vt[col]
    # Normalise for display
    rng = q75-q25 if q75!=q25 else 1
    pos_n = (val-q25)/rng*50+25
    if not lb: pos_n = 100-pos_n
    pos_n = max(2,min(98,pos_n))

    # Coloured band background
    fig_b.add_shape(type="rect",x0=0,x1=25,y0=label,y1=label,
        fillcolor="#D1FAE5",line=dict(width=0),opacity=0.9)
    fig_b.add_shape(type="rect",x0=25,x1=75,y0=label,y1=label,
        fillcolor="#FEF3C7",line=dict(width=0),opacity=0.9)
    fig_b.add_shape(type="rect",x0=75,x1=100,y0=label,y1=label,
        fillcolor="#FEE2E2",line=dict(width=0),opacity=0.9)
    fig_b.add_trace(go.Scatter(x=[pos_n],y=[label],mode="markers+text",
        marker=dict(color="#0A2240",size=14,symbol="line-ns",line=dict(width=3,color="#0A2240")),
        text=[f"{val:.3f}"],textposition="top center",textfont=dict(size=10),
        showlegend=False))

fig_b.update_layout(
    title="VerdaTyres Corp — Industry Band Positioning 2023",
    xaxis=dict(range=[0,100],title="Position in industry (0=worst, 100=best)",tickvals=[25,50,75],
               ticktext=["Bottom 25%","Median","Top 25%"]),
    height=400,plot_bgcolor="#F9FAFB",paper_bgcolor="white",
    shapes=[
        dict(type="rect",x0=0,x1=25,y0=-0.5,y1=4.5,fillcolor="#D1FAE5",opacity=0.15,line_width=0),
        dict(type="rect",x0=25,x1=75,y0=-0.5,y1=4.5,fillcolor="#FEF3C7",opacity=0.15,line_width=0),
        dict(type="rect",x0=75,x1=100,y0=-0.5,y1=4.5,fillcolor="#FEE2E2",opacity=0.15,line_width=0),
    ]
)
fig_b.show()

## 4  Trend — Company vs Industry

In [ ]:
yrs = [str(y) for y in YEARS]
kpi_cols = [("CO2 Intensity","CO2_KPI","#DC2626",1,1),
            ("Energy KPI","Energy_KPI","#7C3AED",1,2),
            ("Water KPI","Water_KPI","#0EA5E9",2,1),
            ("Renew Elec %","Renew_Pct","#00916E",2,2)]

fig_t = make_subplots(2,2,subplot_titles=[t for t,*_ in kpi_cols])
for title,col,color,row,c in kpi_cols:
    vt_trend = df[df.Company=="VerdaTyres Corp"].sort_values("Year")[col].values
    ind_med  = df.groupby("Year")[col].median().values
    fig_t.add_trace(go.Scatter(x=yrs,y=vt_trend,mode="lines+markers",name="VerdaTyres",
        line=dict(color=color,width=2.5),marker=dict(size=4)), row, c)
    fig_t.add_trace(go.Scatter(x=yrs,y=ind_med,mode="lines",name="Industry median",
        line=dict(color="#9CA3AF",width=1.5,dash="dot"),showlegend=(row==1 and c==1)), row, c)

fig_t.update_layout(height=550,title_text="VerdaTyres vs TIP Industry Median 2009-2023",
    plot_bgcolor="white",paper_bgcolor="white")
for i in range(1,3):
    for j in range(1,3):
        fig_t.update_xaxes(tickangle=-45,tickfont=dict(size=8),row=i,col=j)
fig_t.show()

## 5  Improvement Rate Table

In [ ]:
base  = df[df.Year==2009].set_index("Company")
curr  = df[df.Year==2023].set_index("Company")
ind_base = df[df.Year==2009][["CO2_KPI","Energy_KPI","Water_KPI"]].mean()
ind_curr = df[df.Year==2023][["CO2_KPI","Energy_KPI","Water_KPI"]].mean()

cols = [("CO2 intensity","CO2_KPI",True),("Energy intensity","Energy_KPI",True),
        ("Water intensity","Water_KPI",True),("Renew Elec %","Renew_Pct",False),
        ("Waste recovery %","Recovery_Pct",False)]
rows_out = []
for label, col, lb in cols:
    vt_base = base.loc["VerdaTyres Corp",col]
    vt_curr = curr.loc["VerdaTyres Corp",col]
    if col in ind_base:
        ind_b = ind_base[col]; ind_c = ind_curr[col]
    else:
        ind_b = df[df.Year==2009][col].mean(); ind_c = df[df.Year==2023][col].mean()
    vt_chg  = (vt_curr-vt_base)/abs(vt_base)*100 if vt_base else 0
    ind_chg = (ind_c  -ind_b   )/abs(ind_b   )*100 if ind_b   else 0
    lead    = vt_chg-ind_chg if lb else ind_chg-vt_chg
    status  = "Ahead" if (lead<0 and lb) or (lead>0 and not lb) else "Lagging"
    rows_out.append({"KPI":label,
        "VerdaTyres change":f"{vt_chg:+.1f}%",
        "Industry change":  f"{ind_chg:+.1f}%",
        "Lead vs peers":    f"{abs(lead):.1f}pp {'ahead' if status=='Ahead' else 'lagging'}",
        "Status": "Ahead" if status=="Ahead" else "Lagging"})

df_imp = pd.DataFrame(rows_out).set_index("KPI")
styled = df_imp.style.apply(lambda row: [
    "background:#ECFDF5;color:#065F46" if row["Status"]=="Ahead"
    else "background:#FFFBEB;color:#92400E"]*len(row), axis=1)
display(styled)

## 6  Auto-generated Insight Text

In [ ]:
vt_2023 = df[(df.Company=="VerdaTyres Corp")&(df.Year==2023)].iloc[0]
vt_2022 = df[(df.Company=="VerdaTyres Corp")&(df.Year==2022)].iloc[0]
ind_med  = df[df.Year==2023][["CO2_KPI","Energy_KPI","Renew_Pct","Recovery_Pct"]].median()

def chg(a,b): return (a-b)/abs(b)*100 if b else 0

co2_yoy   = chg(vt_2023.CO2_KPI, vt_2022.CO2_KPI)
e_yoy     = chg(vt_2023.Energy_KPI, vt_2022.Energy_KPI)
co2_vs    = chg(vt_2023.CO2_KPI, ind_med.CO2_KPI)
e_vs      = chg(vt_2023.Energy_KPI, ind_med.Energy_KPI)

lines = [
    "="*60,
    "AUTO-GENERATED INSIGHT — VerdaTyres Corp, 2023",
    "NOTE: Review before including in any client report.",
    "="*60,"",
    "CO2 PERFORMANCE",
    f"  KPI: {vt_2023.CO2_KPI:.3f} T/T  |  YoY: {co2_yoy:+.1f}%  |  vs industry: {co2_vs:+.1f}%",
    f"  > CO2 intensity {'improved' if co2_yoy<0 else 'increased'} {abs(co2_yoy):.1f}% year-on-year.",
    f"    At {vt_2023.CO2_KPI:.3f} T/T it is {'below' if co2_vs<0 else 'above'} the TIP median of {ind_med.CO2_KPI:.3f} T/T.",
    "",
    "ENERGY PERFORMANCE",
    f"  KPI: {vt_2023.Energy_KPI:.1f} GJ/T  |  YoY: {e_yoy:+.1f}%  |  vs industry: {e_vs:+.1f}%",
    f"  > Energy intensity {'improved' if e_yoy<0 else 'worsened'} {abs(e_yoy):.1f}% vs prior year.",
    "",
    "RENEWABLE ELECTRICITY",
    f"  {vt_2023.Renew_Pct:.1f}% vs industry median {ind_med.Renew_Pct:.1f}%",
    "",
    "WASTE",
    f"  Recovery rate: {vt_2023.Recovery_Pct:.1f}% vs industry median {ind_med.Recovery_Pct:.1f}%",
]
for l in lines: print(l)

## 7  Export Summary CSV

In [ ]:
summary = df.rename(columns={"Prod_MT":"Production_MT","Energy_GJ":"Total_Energy_GJ",
    "CO2_T":"Total_CO2_T","Water_M3":"Water_M3","Waste_MT":"Waste_Total_MT",
    "Recovery_MT":"Waste_Recovery_MT"})
summary.to_csv("consolidated_summary.csv", index=False)
print(f"Exported consolidated_summary.csv  ({len(summary):,} rows)")
print(f"Companies: {summary.Company.nunique()} | Years: {summary.Year.min()}-{summary.Year.max()}")